In [2]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tifffile import imread
import tifffile as tff

import cv2 as cv
import numpy as np

from pyparsing.common import col

from skimage.segmentation import clear_border

from scipy.spatial import distance

import seaborn as sns

# Utility functions

In [4]:
def display_image(img, fig_size=(20,10), overlay=False, alpha=0.2):
    if overlay:
      plt.figure(figsize=fig_size)
      plt.imshow(img[0], cmap='gray')
      plt.imshow(img[1], cmap='jet', alpha=alpha)
      plt.axis('off')

    else:
      plt.figure(figsize=fig_size)
      plt.imshow(img, cmap='gray')
      plt.axis('off')


def display_image_with_bounding_boxes(image, masks):
  fig, ax = plt.subplots(figsize=(10,10))

  ax.imshow(image)
  ax.axis('off')

  for i in range(len(masks)):
    x,y,w,h = masks[i]['bbox']
    ax.add_patch(plt.Rectangle((x,y), w, h, color='red', fill=False))



def display_image_with_overlay(image, masks, items=None):
  img = np.copy(image)
  img = cv.merge((img, img, img))

  if items is None:
    for i in range(len(masks)):
      contours, _ = cv.findContours(masks[i]['segmentation'].astype('uint8'), cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)
      cv.drawContours(img, contours, -1, color=[255, 0, 0], thickness=3)

  else:
    for i in items:
      contours, _ = cv.findContours(masks[i]['segmentation'].astype('uint8'), cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)
      cv.drawContours(img, contours, -1, color=[255, 0, 0], thickness=3)


  display_image(img)

def read_tiff_and_extract_channels(file_path):
    """
    Reads a TIFF file, extracts protein and lipid channels, and returns 2 image stacks with image patches.

    Args:
        file_path (str): Path to the TIFF file.

    Returns:

    """

    try:
        # Read TIFF file. Image is read in (c x h x w) format
        image = imread(file_path)

        # Check if the image has 6 or 7 channels
        if image.shape[0]==7:
            print("Image has 7 channels")
            image_ch1 = image[1,:,:]
            image_ch2 = image[2,:,:]
            image_ch3 = image[3,:,:]
            image_ch5 = image[5,:,:]

            out_image = [image_ch1, image_ch2, image_ch3, image_ch5]

        elif image.shape[0]==6:
            print("Image has 6 channels")
            image_ch1 = image[0,:,:]
            image_ch2 = image[1,:,:]
            image_ch3 = image[2,:,:]
            image_ch5 = image[4,:,:]

            out_image = [image_ch1, image_ch2, image_ch3, image_ch5]

        else:
            out_image = image
            print("Image has {image.shape[0]} channels")


    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

    return out_image


def preprocess_channel(ch, clipLimit=30):
  clahe = cv.createCLAHE(clipLimit=clipLimit, tileGridSize=(15,15))
  equalized = clahe.apply(image_scaling(ch))
  return image_scaling(equalized)

def image_scaling(image, max=255):
  return (((image-image.min())/(image.max()-image.min()))*max).astype(np.uint8)

